# Week 4. Modeling III: blending, pooling, quality specifications, and the entry to nonlinearity

**Course:** 2105623 Optimization of Chemical Processes
**Institution:** Department of Chemical Engineering, Chulalongkorn University
**Instructor:** Assoc. Prof. Dr. Soorathep Kheawhom

**CLO mapping:** CLO 1 (formulation), CLO 4 (implementation and solution in Pyomo)

## Learning objectives

By the end of this notebook you should be able to:

- Formulate a multi-product blending problem with quality specifications as a linear program, and explain why the specification constraints must be written in unnormalized form.
- Apply the multiply-through-by-the-denominator rule to ratio and quality constraints, state the conditions under which it is exact, and recognize the cases where it fails.
- Explain the structural reason a pool introduces bilinear terms, and write both the p-formulation and the q-formulation of the pooling problem.
- Demonstrate on a concrete instance that a local NLP solver returns different answers from different starting points, and quantify the penalty of accepting a local solution.
- Compare the McCormick relaxations of the two pooling formulations and report the bound each one delivers.

**Estimated duration:** 100 minutes
**Prerequisites:** Weeks 1 to 3, LP formulation and duals, basic notion of a convex set.

**Reference:** Rao, Ch. 1 and Ch. 10; Edgar, Himmelblau and Lasdon, Ch. 7; Haverly (1978) on the pooling problem; Tawarmalani and Sahinidis, *Convexification and Global Optimization*, Ch. 9 (p- and q-formulations).

Discrete and logical decisions (fixed charges, big-M, cardinality, implication) are the subject of Week 5, `W05_modeling4_logical_discrete.ipynb`, and are not treated here.

## Before you start: getting Pyomo and a solver

This is the first session of the course that runs in Python rather than Excel. You do **not** need to install anything on your own machine.

**The route that always works:** open this notebook in **Google Colab** (`File > Upload notebook`) and run the cell below. It takes about two minutes and gives you Pyomo, `ipopt` and HiGHS. Nothing is installed on your laptop and it behaves identically on every machine.

**If you already have a working local install**, run the cell anyway — it will simply confirm what you have and change nothing.

The one-page handout `w04-pyomo-quickstart.pdf` on the course page repeats all of this, plus the three lines of Pyomo you need to read the model listings.


In [ ]:
# --- Run this cell first. -----------------------------------------------
# On Google Colab, or on any machine without a working ipopt, this installs
# Pyomo and the solver binaries and puts them on PATH. It takes about two
# minutes the first time and nothing after that. If your own ipopt already
# works, the cell simply confirms it.
import os, shutil, subprocess, sys

if shutil.which("ipopt") is None and not os.path.exists(
        os.path.expanduser("~/.idaes/bin/ipopt")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyomo", "idaes-pse", "highspy"], check=False)
    subprocess.run(["idaes", "get-extensions"], check=False)

os.environ["PATH"] = os.path.expanduser("~/.idaes/bin") + os.pathsep + os.environ["PATH"]

import pyomo.environ as pyo
print("ipopt   :", pyo.SolverFactory("ipopt").available(exception_flag=False))
print("HiGHS   :", pyo.SolverFactory("appsi_highs").available(exception_flag=False))
print()
print("If both say True you are ready. If ipopt says False, tell the instructor;")
print("you can still do every part of today except Activity 3, which needs an NLP solver.")


In [ ]:
# --- Environment check -------------------------------------------------------
import sys, subprocess, importlib, shutil

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

for p, n in [("pyomo", "pyomo"), ("numpy", "numpy"), ("scipy", "scipy"),
             ("matplotlib", "matplotlib"), ("pandas", "pandas")]:
    ensure(p, n)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pyomo.environ as pyo

def pick_solver(kind="lp"):
    """Return the first available solver of the requested kind."""
    order = {"lp":   ["appsi_highs", "glpk", "cbc", "gurobi", "cplex"],
             "milp": ["appsi_highs", "cbc", "glpk", "gurobi", "cplex"],
             "nlp":  ["ipopt", "conopt", "knitro"],
             "minlp":["bonmin", "couenne", "mindtpy"]}[kind]
    for name in order:
        try:
            s = pyo.SolverFactory(name)
            if s is not None and s.available(exception_flag=False):
                print(f"Using solver: {name}")
                return s
        except Exception:
            continue
    raise RuntimeError(f"No {kind} solver found. Install one, e.g. 'pip install highspy' "
                       f"or 'conda install -c conda-forge ipopt glpk coincbc'.")

In [ ]:
# --- Make locally installed solvers visible ----------------------------------
# The IDAES extensions install ipopt, bonmin and couenne outside the default PATH.
# This cell is a no-op when the binaries are already on PATH.
import os
for extra in [os.path.join(os.path.expanduser("~"), ".idaes", "bin"),
              os.path.join(sys.prefix, "bin")]:
    if os.path.isdir(extra) and extra not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = os.environ.get("PATH", "") + os.pathsep + extra

available = {name: bool(pyo.SolverFactory(name).available(exception_flag=False))
             for name in ["appsi_highs", "glpk", "cbc", "ipopt", "bonmin", "couenne"]}
print("solver availability:", available)

In [ ]:
# --- Figure style: Teal-Amber Lab Palette v1.0 -------------------------------
PALETTE = ["#0F6E6B", "#E29A2D", "#BE654C", "#5A91BE", "#83A462", "#995A90", "#333F4A", "#DFC98F"]
INK, GRAPHITE, MIST, PAPER = "#1C242B", "#333F4A", "#B9C1C6", "#F3F0EB"

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRAPHITE, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.linewidth": 1.0, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": MIST, "grid.linewidth": 0.7, "grid.alpha": 0.9,
    "xtick.color": GRAPHITE, "ytick.color": GRAPHITE,
    "text.color": INK, "lines.linewidth": 1.8, "lines.markersize": 5,
    "font.size": 9, "legend.frameon": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

def tidy(ax):
    """Apply the house style to a single Axes object."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRAPHITE)
    return ax

print("Palette loaded:", ", ".join(PALETTE[:3]), "...")

## 1. Linear blending with quality specifications

A refinery blends five component streams into two gasoline grades. Each component has a purchase cost, an
availability limit, and three quality properties: research octane number (RON), Reid vapor pressure (RVP,
kPa) and sulfur content (ppm). Each finished grade has a minimum RON, a maximum RVP, a maximum sulfur, a
selling price, and a contractual volume window.

### Decision variables

- `x_cg >= 0` tonnes of component `c` sent to grade `g`
- `y_g >= 0` tonnes of grade `g` produced

### Model

    max   sum_g price_g y_g  -  sum_c sum_g cost_c x_cg

    s.t.  sum_c x_cg = y_g                                for all g       (grade material balance)
          sum_g x_cg <= avail_c                           for all c       (component availability)
          dmin_g <= y_g <= dmax_g                         for all g       (contract window)
          sum_c ron_c x_cg >= ron_min_g y_g               for all g       (octane floor)
          sum_c rvp_c x_cg <= rvp_max_g y_g               for all g       (vapor pressure ceiling)
          sum_c sul_c x_cg <= sul_max_g y_g               for all g       (sulfur ceiling)
          x_cg >= 0

### Why the specifications are written this way

The physical requirement is on the *average* property of the blend, for example

    ( sum_c ron_c x_cg ) / ( sum_c x_cg )  >=  ron_min_g

which is a ratio of decision variables and therefore nonlinear. Multiplying through by the denominator, which
is `y_g` and is non-negative, gives the linear form written above. Section 2 states the rule in general and
tests it numerically; section 3 shows the situation in which it stops working.

Two modeling assumptions are hidden here and both should be stated out loud:

1. Properties blend linearly on a mass basis. This is a good approximation for sulfur and for density, an
   acceptable one for RVP if a blending index is used, and only a rough one for octane, where real blending
   is measurably nonlinear.
2. Any component may go to any grade, and the streams are perfectly segregated up to the blender. Section 3
   removes exactly this second assumption, and the model stops being linear.

In [ ]:
# --- Component and grade data ------------------------------------------------
COMPONENTS = ["Reformate", "FCC_Naphtha", "Alkylate", "Butane", "Straight_Run"]
GRADES = ["Regular", "Premium"]

comp = pd.DataFrame({
    "ron":       [98.0,  92.0,  95.0,  93.0,  68.0],
    "rvp_kpa":   [ 3.0,   6.5,   4.5,  62.0,   9.0],
    "sulfur_ppm":[110.0, 380.0,  15.0,   5.0, 520.0],
    "cost_usd_t":[760.0, 730.0, 805.0, 510.0, 695.0],
    "avail_t":   [3200.0, 4000.0, 2000.0, 900.0, 2600.0],
}, index=COMPONENTS)

grade = pd.DataFrame({
    "ron_min":    [91.0,  95.0],
    "rvp_max":    [ 9.0,   8.5],
    "sulfur_max": [290.0, 190.0],
    "price_usd_t":[742.0, 806.0],
    "vol_min_t":  [4000.0, 2500.0],
    "vol_max_t":  [7000.0, 5000.0],
}, index=GRADES)

print("components\n", comp, "\n")
print("finished grades\n", grade)

In [ ]:
# --- Blending LP -------------------------------------------------------------
def build_blend(duals=True):
    m = pyo.ConcreteModel(name="linear_blending")
    m.C = pyo.Set(initialize=COMPONENTS)
    m.G = pyo.Set(initialize=GRADES)

    m.x = pyo.Var(m.C, m.G, domain=pyo.NonNegativeReals)
    m.y = pyo.Var(m.G, domain=pyo.NonNegativeReals)

    m.grade_balance = pyo.Constraint(
        m.G, rule=lambda m, g: sum(m.x[c, g] for c in m.C) == m.y[g])
    m.availability = pyo.Constraint(
        m.C, rule=lambda m, c: sum(m.x[c, g] for g in m.G) <= comp.avail_t[c])
    m.vol_lo = pyo.Constraint(m.G, rule=lambda m, g: m.y[g] >= grade.vol_min_t[g])
    m.vol_hi = pyo.Constraint(m.G, rule=lambda m, g: m.y[g] <= grade.vol_max_t[g])

    m.octane = pyo.Constraint(
        m.G, rule=lambda m, g: sum(comp.ron[c]*m.x[c, g] for c in m.C)
                               >= grade.ron_min[g]*m.y[g])
    m.vapor = pyo.Constraint(
        m.G, rule=lambda m, g: sum(comp.rvp_kpa[c]*m.x[c, g] for c in m.C)
                               <= grade.rvp_max[g]*m.y[g])
    m.sulfur = pyo.Constraint(
        m.G, rule=lambda m, g: sum(comp.sulfur_ppm[c]*m.x[c, g] for c in m.C)
                               <= grade.sulfur_max[g]*m.y[g])

    m.profit = pyo.Objective(
        expr=sum(grade.price_usd_t[g]*m.y[g] for g in m.G)
             - sum(comp.cost_usd_t[c]*m.x[c, g] for c in m.C for g in m.G),
        sense=pyo.maximize)
    if duals:
        m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)   # only meaningful for the LP
    return m

lp = pick_solver("lp")
mb = build_blend()
rb = lp.solve(mb)
print(rb.solver.termination_condition)
assert rb.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "blending LP did not solve to optimality"
PROFIT_LP = pyo.value(mb.profit)
print(f"maximum blending margin: {PROFIT_LP:,.2f} USD")

---

### The same model in the spreadsheet

The companion workbook for this week is `excel/W04_blending_STUDENT.xlsx`, with the
completed version in `excel/W04_blending_SOLUTION.xlsx`. Per the tool allocation table
of the course specification, Week 4 is **Excel then Python, and it is the designed crossover point of
the course**: the linear blending model below is built in OpenSolver, and the pooling problem of Section
3 forces the move to Python with ipopt and a multistart. On the `Model` sheet the two grades occupy
columns `C` and `D` throughout, so every quality family is one row per property with two columns.

| Algebraic symbol | Spreadsheet range or layout | Pyomo component | Note |
|---|---|---|---|
| `c in C`, components | row labels `A20:A24` of the `Solution` block, matching `A5:A9` in the data block | `m.C = pyo.Set(initialize=COMPONENTS)` | The sheet repeats the index labels on every block; Pyomo declares the set once and reuses it. |
| `g in G`, finished grades | column headers `C10:D10` and `C19:D19` | `m.G = pyo.Set(initialize=GRADES)` | Two columns for two grades. A third grade is a third column in five separate blocks. |
| `ron_c`, `rvp_c`, `sul_c` | data columns `B5:B9`, `C5:C9`, `D5:D9`, blue text | the `comp` DataFrame, read directly inside the constraint rules | Week 4 reads the DataFrame rather than declaring `pyo.Param`, which is legitimate for a `ConcreteModel`. The fully declared alternative is `pyo.Param(m.C, initialize=comp.ron.to_dict())`. |
| `cost_c`, purchase cost | data column `E5:E9`, blue text | `comp.cost_usd_t` in the `m.profit` expression | |
| `avail_c`, availability | `RHS` column `G20:G24`, each cell linked by `=F5` and copied down | `comp.avail_t` in `m.availability` | Linking rather than retyping is the rule: no computed cell may hold a hardcoded number. |
| `price_g`, `ron_min_g`, `rvp_max_g`, `sul_max_g`, `dmin_g`, `dmax_g` | grade data block `C11:D16`, blue text, one row per property | the `grade` DataFrame | Grade data is transposed relative to component data, because grades index columns and components index rows. |
| `x_cg >= 0`, component to grade | changing cells `C20:D24`, green fill, 5 by 2 | `m.x = pyo.Var(m.C, m.G, domain=pyo.NonNegativeReals)` | |
| `y_g >= 0`, grade produced | changing cells `C26:D26`, green fill | `m.y = pyo.Var(m.G, domain=pyo.NonNegativeReals)` | |
| `sum_c x_cg = y_g` | `VALUE` row `C28:D28`, `=SUM(C20:C24)-C26`; relation row `C29:D29` holding `=`; `RHS` row `C30:D30` of zeros | `m.grade_balance = pyo.Constraint(m.G, rule=...)` | Writing the balance as `VALUE = 0` keeps a variable out of the `RHS` column wherever that is possible. |
| `sum_g x_cg <= avail_c` | `VALUE` column `E20:E24`, `=SUM(C20:D20)`; relation column `F20:F24`; `RHS` column `G20:G24` | `m.availability = pyo.Constraint(m.C, rule=...)` | The row sums of the changing block, exactly as in the Week 3 transportation sheet. |
| `dmin_g <= y_g <= dmax_g` | dialog entries `$C$26:$D$26 >= $C$15:$D$15` and `$C$26:$D$26 <= $C$16:$D$16` | `m.vol_lo`, `m.vol_hi` | Two dialog lines, two `Constraint` components, one contract window. |
| `sum_c ron_c x_cg >= ron_min_g y_g` | `VALUE` row `C32:D32`, `=SUMPRODUCT($B$5:$B$9,C20:C24)`; relation row `C33:D33` holding `>=`; `RHS` row `C34:D34`, `=C11*C26` | `m.octane = pyo.Constraint(m.G, rule=...)` | The `RHS` cell now contains a changing cell. This is what "multiply the ratio through by its denominator" looks like on a sheet, and the plain `RHS` column convention does not anticipate it. |
| `sum_c rvp_c x_cg <= rvp_max_g y_g` | `VALUE` row `C36:D36`; relation row `C37:D37`; `RHS` row `C38:D38`, `=C12*C26` | `m.vapor = pyo.Constraint(m.G, rule=...)` | |
| `sum_c sul_c x_cg <= sul_max_g y_g` | `VALUE` row `C40:D40`; relation row `C41:D41`; `RHS` row `C42:D42`, `=C13*C26` | `m.sulfur = pyo.Constraint(m.G, rule=...)` | |
| `max sum_g price_g y_g - sum_c sum_g cost_c x_cg` | objective cell `C45`, `=SUMPRODUCT(C14:D14,C26:D26)-SUMPRODUCT($E$5:$E$9,E20:E24)`, entered with To: Max | `m.profit = pyo.Objective(expr=..., sense=pyo.maximize)` | The second term reuses the availability `VALUE` column, so total usage is computed once. |
| `p`, pool quality, and the bilinear terms `p P_g` | **no layout exists.** The pool sulfur balance becomes a cell such as `=C48*C49`, a product of two changing cells | `m.p = pyo.Var(bounds=(1.0, 3.0))`, `m.pool_sulfur`, `m.spec_X`, `m.spec_Y` | The sheet accepts the formula without comment and silently changes the class of the problem. This row is the crossover. |

**Where the spreadsheet stops working.** The linear model of Section 1 is a good spreadsheet: 12
changing cells, 5 availability rows, 2 balance rows and 6 specification rows, every one of them a
`SUMPRODUCT`, and Simplex LP reproduces the margin printed above. The single physical change in Section
3 is that the components now reach the blender through a shared pool, so the sulfur delivered to a
product is a pool quality times a flow, and one cell of the sheet multiplies two changing cells. Excel
accepts it. Solver quietly abandons Simplex LP for GRG Nonlinear, converges, and reports "Solver found a
solution. All constraints and optimality conditions are satisfied." That sentence is true and
misleading: GRG certifies a **local** optimum, and the two starting points of Section 3 reach two
different profits on identical data, only one of which the global solver of Section 4 certifies. Nothing
in the dialog, on the sheet or in the report distinguishes them. There is no bound, no gap and no
warning, so a plant engineer who blends to the sheet's recipe has no evidence that a better one exists.
Running a proper multistart means retyping starting values into the changing cells and pressing Solve
once per start, which is why Exercise 4 asks for 100 starts in a loop; and the McCormick relaxation of
Section 4 adds four inequality rows for every bilinear term, changing the shape of the model rather than
its data. The course crosses to Python here, and the reason is not size. It is that a spreadsheet solver
cannot tell you that its answer is local.

---

In [ ]:
# --- Blend recipe and realized qualities -------------------------------------
recipe = pd.DataFrame([[pyo.value(mb.x[c, g]) for g in GRADES] for c in COMPONENTS],
                      index=COMPONENTS, columns=GRADES).round(3)
recipe["used_t"] = recipe.sum(axis=1)
recipe["avail_t"] = comp.avail_t
recipe["slack_t"] = (recipe.avail_t - recipe.used_t).round(3)
print(recipe, "\n")

qual = []
for g in GRADES:
    Y = pyo.value(mb.y[g])
    qual.append({
        "grade": g, "volume_t": round(Y, 3),
        "RON":    round(sum(comp.ron[c]*pyo.value(mb.x[c, g]) for c in COMPONENTS)/Y, 4),
        "RON_min": grade.ron_min[g],
        "RVP":    round(sum(comp.rvp_kpa[c]*pyo.value(mb.x[c, g]) for c in COMPONENTS)/Y, 4),
        "RVP_max": grade.rvp_max[g],
        "S_ppm":  round(sum(comp.sulfur_ppm[c]*pyo.value(mb.x[c, g]) for c in COMPONENTS)/Y, 3),
        "S_max":  grade.sulfur_max[g]})
qual = pd.DataFrame(qual).set_index("grade")
print(qual)

for g in GRADES:
    assert qual.RON[g] >= grade.ron_min[g] - 1e-6
    assert qual.RVP[g] <= grade.rvp_max[g] + 1e-6
    assert qual.S_ppm[g] <= grade.sulfur_max[g] + 1e-6
print("\nall quality specifications satisfied")

In [ ]:
# --- Marginal values ---------------------------------------------------------
mv = pd.DataFrame({
    "availability_dual": [mb.dual[mb.availability[c]] for c in COMPONENTS],
    "purchase_cost":     [comp.cost_usd_t[c] for c in COMPONENTS],
}, index=COMPONENTS).round(4)
mv["breakeven_price"] = (mv.purchase_cost + mv.availability_dual).round(4)
print("value of one extra tonne of each component:")
print(mv, "\n")

specs = pd.DataFrame({
    "octane_dual": [mb.dual[mb.octane[g]] for g in GRADES],
    "vapor_dual":  [mb.dual[mb.vapor[g]] for g in GRADES],
    "sulfur_dual": [mb.dual[mb.sulfur[g]] for g in GRADES],
}, index=GRADES).round(5)
print("value of relaxing each specification by one unit:")
print(specs)

In [ ]:
# --- Figure 1: recipe and specification utilization --------------------------
fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6), constrained_layout=True)

ax = tidy(axes[0])
bottom = np.zeros(len(GRADES))
for k, c in enumerate(COMPONENTS):
    vals = np.array([pyo.value(mb.x[c, g]) for g in GRADES])
    ax.bar(GRADES, vals, bottom=bottom, color=PALETTE[k], label=c.replace("_", " "))
    for i, v in enumerate(vals):
        if v > 300:
            ax.text(i, bottom[i] + v/2, f"{v:,.0f}", ha="center", va="center",
                    fontsize=7, color="white")
    bottom += vals
ax.set_ylabel("tonnes"); ax.set_title("Optimal blend recipe", fontsize=10)
ax.legend(fontsize=7.5, ncol=2, loc="upper left")
ax.set_ylim(0, bottom.max()*1.38)

ax = tidy(axes[1])
ypos = np.arange(len(COMPONENTS))[::-1]
used = np.array([recipe.used_t[c] for c in COMPONENTS])
avail = comp.avail_t.values
ax.barh(ypos, avail, color=MIST, height=0.62, label="available")
ax.barh(ypos, used, color=PALETTE[0], height=0.62, label="used at the optimum")
for k, c in enumerate(COMPONENTS):
    d = mb.dual[mb.availability[c]]
    col = PALETTE[1] if d > 1e-6 else GRAPHITE
    ax.text(avail[k] + 90, ypos[k], f"dual {d:,.2f} USD/t", va="center",
            fontsize=7.2, color=col)
ax.set_yticks(ypos); ax.set_yticklabels([c.replace("_", " ") for c in COMPONENTS], fontsize=8)
ax.set_xlabel("tonnes"); ax.set_xlim(0, 6400)
ax.set_title("Component use and its marginal value", fontsize=10)
ax.legend(fontsize=8, loc="center right")

fig.suptitle(f"Week 4, Figure 1: linear blending optimum, {PROFIT_LP:,.0f} USD "
             f"(all six quality specifications bind)",
             color=INK, fontsize=10)
plt.show()

## 2. Linearizing ratio and quality constraints

### 2.1 The rule

Let `a` and `b` be vectors of data and `x` the decision vector. A requirement of the form

    ( a' x ) / ( b' x )  <=  beta

is equivalent to the linear inequality

    a' x  -  beta ( b' x )  <=  0

**provided that `b' x > 0` at every point of interest.** The equivalence has three separate conditions and
each of them fails somewhere in practice:

1. **Sign.** Multiplying an inequality by a negative number reverses it. The rule as written needs
   `b' x > 0`, not merely `b' x != 0`. In blending the denominator is a total mass and is non-negative by
   construction, and the contract window forces it strictly positive.
2. **Zero denominator.** If `b' x = 0` is attainable, the ratio is undefined while the multiplied-through
   form is satisfied trivially (`0 <= 0`). The linear model is then a strict relaxation of the intended one:
   it permits a blend of size zero whose quality is meaningless. If a grade may be shut off, the correct
   model is a disjunction and needs a binary variable, which is Week 5 material.
3. **Same balance.** The denominator must be a linear function of the *same* variables that appear in the
   numerator, so that `beta ( b' x )` is linear. When the denominator belongs to a different balance, as in a
   pool, the product of two decision variables survives and the model is genuinely nonlinear.

### 2.2 Translation table

| Engineering requirement | Natural form | Linear equivalent | Condition |
|---|---|---|---|
| average property of a blend at most `beta` | `sum_c q_c x_c / sum_c x_c <= beta` | `sum_c (q_c - beta) x_c <= 0` | `sum_c x_c > 0` |
| average property at least `beta` | `sum_c q_c x_c / sum_c x_c >= beta` | `sum_c (q_c - beta) x_c >= 0` | `sum_c x_c > 0` |
| recycle ratio at most `r` | `R / (R + F) <= r` | `(1 - r) R - r F <= 0` | `R + F > 0` |
| conversion at least `eta` | `(F_in - F_out) / F_in >= eta` | `F_out <= (1 - eta) F_in` | `F_in > 0` |
| fixed split fraction `theta` | `S_1 / (S_1 + S_2) = theta` | `(1 - theta) S_1 - theta S_2 = 0` | `S_1 + S_2 > 0` |
| purity of a mixed stream at least `pi` | `m_key / m_total >= pi` | `m_key - pi m_total >= 0` | `m_total > 0` |
| ratio *objective* `max (c'x + c0)/(d'x + d0)` | linear-fractional | Charnes-Cooper LP, section 2.4 | `d'x + d0 > 0` |
| property of a stream leaving a **pool** | `p = sum_i q_i F_i / sum_g P_g`, and `p` multiplies `P_g` | **no linear equivalent** | denominator belongs to another balance |

The last row is the one that matters for the rest of this notebook. The practical test is a single question:
*is the denominator the same aggregate that the numerator is normalized by, in the same balance?* If yes, the
constraint is linear in disguise. If no, expect bilinear terms.

In [ ]:
# --- The ratio form and the linear form give the same optimum ----------------
def build_blend_ratio():
    """Blending model with the quality specifications left as explicit ratios."""
    m = pyo.ConcreteModel(name="blending_ratio_form")
    m.C = pyo.Set(initialize=COMPONENTS)
    m.G = pyo.Set(initialize=GRADES)
    m.x = pyo.Var(m.C, m.G, domain=pyo.NonNegativeReals, initialize=500.0)
    m.y = pyo.Var(m.G, domain=pyo.NonNegativeReals,
                  initialize=lambda m, g: float(grade.vol_min_t[g]))

    m.grade_balance = pyo.Constraint(m.G, rule=lambda m, g: sum(m.x[c, g] for c in m.C) == m.y[g])
    m.availability  = pyo.Constraint(m.C, rule=lambda m, c: sum(m.x[c, g] for g in m.G) <= comp.avail_t[c])
    m.vol_lo = pyo.Constraint(m.G, rule=lambda m, g: m.y[g] >= grade.vol_min_t[g])
    m.vol_hi = pyo.Constraint(m.G, rule=lambda m, g: m.y[g] <= grade.vol_max_t[g])

    m.octane = pyo.Constraint(m.G, rule=lambda m, g:
        sum(comp.ron[c]*m.x[c, g] for c in m.C) / m.y[g] >= grade.ron_min[g])
    m.vapor  = pyo.Constraint(m.G, rule=lambda m, g:
        sum(comp.rvp_kpa[c]*m.x[c, g] for c in m.C) / m.y[g] <= grade.rvp_max[g])
    m.sulfur = pyo.Constraint(m.G, rule=lambda m, g:
        sum(comp.sulfur_ppm[c]*m.x[c, g] for c in m.C) / m.y[g] <= grade.sulfur_max[g])

    m.profit = pyo.Objective(
        expr=sum(grade.price_usd_t[g]*m.y[g] for g in m.G)
             - sum(comp.cost_usd_t[c]*m.x[c, g] for c in m.C for g in m.G),
        sense=pyo.maximize)
    return m

nlp = pick_solver("nlp")
mr = build_blend_ratio()
print("polynomial degree of the ratio constraint:",
      pyo.polynomial_degree(mr.octane["Regular"].body), "(None means not polynomial)")
print("polynomial degree of the linear constraint:",
      pyo.polynomial_degree(mb.octane["Regular"].body))

rr = nlp.solve(mr)
assert rr.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "ipopt did not converge on the ratio form"
PROFIT_RATIO = pyo.value(mr.profit)
print(f"\nlinear form solved by {'HiGHS/CBC/GLPK'} : {PROFIT_LP:>14,.4f} USD")
print(f"ratio  form solved by ipopt          : {PROFIT_RATIO:>14,.4f} USD")
print(f"difference                           : {abs(PROFIT_LP - PROFIT_RATIO):>14,.4f} USD")
assert abs(PROFIT_LP - PROFIT_RATIO) < 1.0, "the two forms should describe the same optimum"

# the zero-denominator caveat, made concrete
print("\nAt a shut-off grade (y_g = 0, all x_cg = 0):")
print("  linear form   0 >= 91 * 0  ->", 0.0 >= 91.0*0.0, " (satisfied, and vacuous)")
try:
    _ = 0.0 / 0.0
except ZeroDivisionError as exc:
    print("  ratio  form   0 / 0        ->", type(exc).__name__, f"({exc})")
print("The two models differ exactly on the set where the grade is not produced.")

### 2.3 A ratio in the objective: the Charnes-Cooper transformation

Ratio *objectives* also occur: margin per tonne, cost per unit of product, energy per unit of hydrogen,
carbon intensity. Maximizing

    ( c' x + c0 ) / ( d' x + d0 )

over a polyhedron `{ A x <= b, x >= 0 }` with `d' x + d0 > 0` is a **linear-fractional program**. It is not an
LP, but it becomes one after the Charnes-Cooper change of variables

    t = 1 / ( d' x + d0 ) > 0,        x' = t x

which gives the equivalent linear program

    max   c' x' + c0 t
    s.t.  A x' - b t <= 0
          d' x' + d0 t = 1
          x' >= 0,  t >= 0

and the original solution is recovered as `x = x' / t`. Every constraint acquires the variable `t` in place of
its right-hand side: constraints that were homogeneous of degree one, such as the quality specifications
above, are unchanged.

The important modeling point is that a ratio objective is a **different objective**, not a rescaling of the
original one. Maximizing margin per tonne and maximizing total margin generally give different plans.

In [ ]:
# --- Maximize margin per tonne: fractional NLP versus Charnes-Cooper LP ------
def build_charnes_cooper():
    """LP equivalent of  max (revenue - cost) / total volume."""
    m = pyo.ConcreteModel(name="margin_per_tonne_LP")
    m.C = pyo.Set(initialize=COMPONENTS)
    m.G = pyo.Set(initialize=GRADES)
    m.xs = pyo.Var(m.C, m.G, domain=pyo.NonNegativeReals)   # x' = t x
    m.ys = pyo.Var(m.G, domain=pyo.NonNegativeReals)        # y' = t y
    m.t  = pyo.Var(domain=pyo.NonNegativeReals)             # t = 1 / total volume

    m.grade_balance = pyo.Constraint(m.G, rule=lambda m, g: sum(m.xs[c, g] for c in m.C) == m.ys[g])
    m.availability  = pyo.Constraint(m.C, rule=lambda m, c:
        sum(m.xs[c, g] for g in m.G) <= comp.avail_t[c]*m.t)
    m.vol_lo = pyo.Constraint(m.G, rule=lambda m, g: m.ys[g] >= grade.vol_min_t[g]*m.t)
    m.vol_hi = pyo.Constraint(m.G, rule=lambda m, g: m.ys[g] <= grade.vol_max_t[g]*m.t)
    m.octane = pyo.Constraint(m.G, rule=lambda m, g:
        sum(comp.ron[c]*m.xs[c, g] for c in m.C) >= grade.ron_min[g]*m.ys[g])
    m.vapor  = pyo.Constraint(m.G, rule=lambda m, g:
        sum(comp.rvp_kpa[c]*m.xs[c, g] for c in m.C) <= grade.rvp_max[g]*m.ys[g])
    m.sulfur = pyo.Constraint(m.G, rule=lambda m, g:
        sum(comp.sulfur_ppm[c]*m.xs[c, g] for c in m.C) <= grade.sulfur_max[g]*m.ys[g])
    m.normalize = pyo.Constraint(expr=sum(m.ys[g] for g in m.G) == 1.0)   # d'x' + d0 t = 1

    m.rate = pyo.Objective(
        expr=sum(grade.price_usd_t[g]*m.ys[g] for g in m.G)
             - sum(comp.cost_usd_t[c]*m.xs[c, g] for c in m.C for g in m.G),
        sense=pyo.maximize)
    return m

mcc = build_charnes_cooper()
rcc = lp.solve(mcc)
assert rcc.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "Charnes-Cooper LP did not solve"
T = pyo.value(mcc.t)
RATE_LP = pyo.value(mcc.rate)
y_cc = {g: pyo.value(mcc.ys[g])/T for g in GRADES}
margin_cc = sum(grade.price_usd_t[g]*y_cc[g] for g in GRADES) \
            - sum(comp.cost_usd_t[c]*pyo.value(mcc.xs[c, g])/T for c in COMPONENTS for g in GRADES)

# the same objective attacked directly as a nonlinear program
mfr = build_blend_ratio()
mfr.del_component(mfr.profit)
mfr.rate = pyo.Objective(
    expr=(sum(grade.price_usd_t[g]*mfr.y[g] for g in mfr.G)
          - sum(comp.cost_usd_t[c]*mfr.x[c, g] for c in mfr.C for g in mfr.G))
         / sum(mfr.y[g] for g in mfr.G), sense=pyo.maximize)
rfr = nlp.solve(mfr)
assert rfr.solver.termination_condition == pyo.TerminationCondition.optimal

cc = pd.DataFrame({
    "objective": ["total margin (section 1 LP)", "margin per tonne (Charnes-Cooper LP)",
                  "margin per tonne (direct NLP)"],
    "value": [f"{PROFIT_LP:,.4f} USD", f"{RATE_LP:,.6f} USD/t", f"{pyo.value(mfr.rate):,.6f} USD/t"],
    "Regular t": [round(pyo.value(mb.y['Regular']), 3), round(y_cc['Regular'], 3),
                  round(pyo.value(mfr.y['Regular']), 3)],
    "Premium t": [round(pyo.value(mb.y['Premium']), 3), round(y_cc['Premium'], 3),
                  round(pyo.value(mfr.y['Premium']), 3)],
    "total margin USD": [round(PROFIT_LP, 3), round(margin_cc, 3),
                         round(pyo.value(mfr.rate)*sum(pyo.value(mfr.y[g]) for g in GRADES), 3)]})
print(cc.to_string(index=False))
print(f"\nt = {T:.8f} = 1 / {1/T:,.3f} t of total production")
assert abs(RATE_LP - pyo.value(mfr.rate)) < 1e-4, \
    "the Charnes-Cooper LP and the direct fractional NLP must agree"
print(f"\nMaximizing the ratio gives up {PROFIT_LP - margin_cc:,.3f} USD of total margin "
      f"to raise the margin per tonne to {RATE_LP:.4f} USD/t.")

## 3. The pooling problem: where bilinear terms come from

The blending model above assumes every component reaches the blender as a separate stream. Real plants do not
have that many tanks. Streams are often collected in a common **pool**, and everything drawn from that pool
has the same composition, whatever it is fed by.

That single physical fact destroys linearity, and it does so through the third condition of section 2.1. Let
`F_i` be the flows into a pool fed by streams of qualities `q_i`, and let the pool deliver `P_g` to each
product `g`. The pool quality `p` satisfies

    p * ( sum_g P_g )  =  sum_i q_i F_i

and the quality delivered to product `g` is `p * P_g`. Both expressions contain the product of two decision
variables, `p` and a flow. Multiplying the ratio by its denominator does not help here: the denominator is the
total pool throughput, while the quantity that has to be constrained is the contribution of the pool to the
balance of a *different* stream, namely product `g`. The feasible set is no longer convex, and the problem is
a nonconvex NLP with bilinear terms.

### Haverly's instance

The standard minimal example (Haverly, 1978) is small enough to reason about by hand.

- Feed `A`: sulfur 3.0 percent, cost 6 USD/unit, into the pool.
- Feed `B`: sulfur 1.0 percent, cost 16 USD/unit, into the pool.
- Feed `C`: sulfur 2.0 percent, cost 10 USD/unit, bypasses the pool.
- Product `X`: sulfur at most 2.5 percent, price 9 USD/unit, at most 100 units.
- Product `Y`: sulfur at most 1.5 percent, price 15 USD/unit, at most 200 units.

Variables: `A`, `B` into the pool, `Px`, `Py` out of the pool to each product, `Cx`, `Cy` direct, and the
pool sulfur `p`. This is the **p-formulation**, so called because the pool quality itself is a variable.

    max   9(Px + Cx) + 15(Py + Cy) - 6A - 16B - 10(Cx + Cy)

    s.t.  A + B = Px + Py                          (pool mass balance)
          3A + 1B = p (Px + Py)                    (pool sulfur balance, bilinear)
          p Px + 2 Cx <= 2.5 (Px + Cx)             (product X sulfur, bilinear)
          p Py + 2 Cy <= 1.5 (Py + Cy)             (product Y sulfur, bilinear)
          Px + Cx <= 100,   Py + Cy <= 200
          1 <= p <= 3,      all flows >= 0

The bounds on `p` are physical: the pool is fed only by streams of 1 and 3 percent sulfur, so its quality
must lie between them.

In [ ]:
# --- Haverly pooling model ---------------------------------------------------
def build_haverly(p0=2.0, A0=0.0, B0=0.0, Px0=0.0, Py0=0.0, Cx0=0.0, Cy0=0.0):
    m = pyo.ConcreteModel(name="haverly_pooling")
    m.A  = pyo.Var(domain=pyo.NonNegativeReals, initialize=A0)
    m.B  = pyo.Var(domain=pyo.NonNegativeReals, initialize=B0)
    m.Px = pyo.Var(domain=pyo.NonNegativeReals, initialize=Px0)
    m.Py = pyo.Var(domain=pyo.NonNegativeReals, initialize=Py0)
    m.Cx = pyo.Var(domain=pyo.NonNegativeReals, initialize=Cx0)
    m.Cy = pyo.Var(domain=pyo.NonNegativeReals, initialize=Cy0)
    m.p  = pyo.Var(bounds=(1.0, 3.0), initialize=p0)          # pool sulfur, percent

    m.pool_mass    = pyo.Constraint(expr=m.A + m.B == m.Px + m.Py)
    m.pool_sulfur  = pyo.Constraint(expr=3*m.A + 1*m.B == m.p*(m.Px + m.Py))
    m.spec_X       = pyo.Constraint(expr=m.p*m.Px + 2*m.Cx <= 2.5*(m.Px + m.Cx))
    m.spec_Y       = pyo.Constraint(expr=m.p*m.Py + 2*m.Cy <= 1.5*(m.Py + m.Cy))
    m.demand_X     = pyo.Constraint(expr=m.Px + m.Cx <= 100)
    m.demand_Y     = pyo.Constraint(expr=m.Py + m.Cy <= 200)

    m.profit = pyo.Objective(
        expr=9*(m.Px + m.Cx) + 15*(m.Py + m.Cy) - 6*m.A - 16*m.B - 10*(m.Cx + m.Cy),
        sense=pyo.maximize)
    return m

nlp = pick_solver("nlp")
print("degree of the pooling constraints:",
      {"pool_sulfur": pyo.polynomial_degree(build_haverly().pool_sulfur.body),
       "spec_X":      pyo.polynomial_degree(build_haverly().spec_X.body)})

In [ ]:
# --- The same NLP from two different starting points -------------------------
def run_from(p0, label):
    m = build_haverly(p0=p0)
    r = nlp.solve(m, tee=False)
    assert r.solver.termination_condition == pyo.TerminationCondition.optimal, \
        f"ipopt did not converge from p0 = {p0}"
    return {"label": label, "p_start": p0,
            "status": str(r.solver.termination_condition),
            "profit": round(pyo.value(m.profit), 6),
            "p_final": round(pyo.value(m.p), 6),
            "A": round(pyo.value(m.A), 4), "B": round(pyo.value(m.B), 4),
            "Px": round(pyo.value(m.Px), 4), "Py": round(pyo.value(m.Py), 4),
            "Cx": round(pyo.value(m.Cx), 4), "Cy": round(pyo.value(m.Cy), 4)}

low  = run_from(1.0, "start at the low-sulfur end")
high = run_from(3.0, "start at the high-sulfur end")

print(pd.DataFrame([low, high]).set_index("label").T.to_string())
print()
print(f"Both runs report '{low['status']}'. The two answers differ by "
      f"{low['profit'] - high['profit']:.1f} USD, which is "
      f"{100*(low['profit'] - high['profit'])/low['profit']:.0f} percent of the better one.")
assert low["profit"] > high["profit"] + 1.0, "expected two distinct local solutions"

In [ ]:
# --- Multistart: which starting points find which solution -------------------
starts = np.linspace(1.0, 3.0, 9)
ms = []
for p0 in starts:
    m = build_haverly(p0=float(p0))
    r = nlp.solve(m, tee=False)
    assert r.solver.termination_condition == pyo.TerminationCondition.optimal,         f"ipopt failed from p0 = {p0}"
    ms.append({"p_start": round(float(p0), 3),
               "termination": str(r.solver.termination_condition),
               "profit": round(pyo.value(m.profit), 4),
               "p_final": round(pyo.value(m.p), 4)})
ms = pd.DataFrame(ms)
print(ms.to_string(index=False))
print("\ndistinct local optima found:", sorted(ms.profit.unique(), reverse=True))
print(f"fraction of starting points that reach the better solution: "
      f"{100*(ms.profit == ms.profit.max()).mean():.0f} percent")

In [ ]:
# --- Why: profit as a function of the pool quality ---------------------------
# For a FIXED value of p the pooling problem is an ordinary LP. Sweeping p therefore
# traces the true profit profile, and any local solver can only climb the hill it starts on.
def profit_at_fixed_p(p):
    m = pyo.ConcreteModel()
    m.A  = pyo.Var(domain=pyo.NonNegativeReals); m.B  = pyo.Var(domain=pyo.NonNegativeReals)
    m.Px = pyo.Var(domain=pyo.NonNegativeReals); m.Py = pyo.Var(domain=pyo.NonNegativeReals)
    m.Cx = pyo.Var(domain=pyo.NonNegativeReals); m.Cy = pyo.Var(domain=pyo.NonNegativeReals)
    m.pool_mass   = pyo.Constraint(expr=m.A + m.B == m.Px + m.Py)
    m.pool_sulfur = pyo.Constraint(expr=3*m.A + 1*m.B == p*(m.Px + m.Py))
    m.spec_X      = pyo.Constraint(expr=p*m.Px + 2*m.Cx <= 2.5*(m.Px + m.Cx))
    m.spec_Y      = pyo.Constraint(expr=p*m.Py + 2*m.Cy <= 1.5*(m.Py + m.Cy))
    m.demand_X    = pyo.Constraint(expr=m.Px + m.Cx <= 100)
    m.demand_Y    = pyo.Constraint(expr=m.Py + m.Cy <= 200)
    m.profit = pyo.Objective(
        expr=9*(m.Px + m.Cx) + 15*(m.Py + m.Cy) - 6*m.A - 16*m.B - 10*(m.Cx + m.Cy),
        sense=pyo.maximize)
    r = lp.solve(m)
    return (pyo.value(m.profit)
            if r.solver.termination_condition == pyo.TerminationCondition.optimal else np.nan)

pgrid = np.linspace(1.0, 3.0, 81)
pprof = np.array([profit_at_fixed_p(float(p)) for p in pgrid])
best_i = int(np.nanargmax(pprof))
print(f"best profit on the grid: {pprof[best_i]:.4f} at p = {pgrid[best_i]:.4f}")
print(f"profit at p = 3.0      : {pprof[-1]:.4f}")
print(f"flat zero region       : p in [{pgrid[pprof < 1e-6].min():.3f}, "
      f"{pgrid[pprof < 1e-6].max():.3f}]")

In [ ]:
# --- Figure 2: the nonconvex profit landscape --------------------------------
fig, ax = plt.subplots(figsize=(6.4, 3.8), constrained_layout=True)
tidy(ax)
ax.fill_between(pgrid, 0, pprof, color=PALETTE[7], alpha=0.55)
ax.plot(pgrid, pprof, color=PALETTE[0], lw=2.0, label="optimal profit for a fixed pool quality")
ax.plot([low["p_final"]], [low["profit"]], marker="o", ms=9, color=PALETTE[1],
        ls="none", label=f"global optimum, profit {low['profit']:.0f}")
ax.plot([high["p_final"]], [high["profit"]], marker="s", ms=8, color=PALETTE[2],
        ls="none", label=f"local optimum, profit {high['profit']:.0f}")
ax.annotate("ipopt started here\nclimbs to the left peak", xy=(1.0, low["profit"]),
            xytext=(1.35, 340), fontsize=8, color=INK,
            arrowprops=dict(arrowstyle="->", color=GRAPHITE, lw=1.2))
ax.annotate("started here\nclimbs to the right peak", xy=(3.0, high["profit"]),
            xytext=(2.15, 190), fontsize=8, color=INK,
            arrowprops=dict(arrowstyle="->", color=GRAPHITE, lw=1.2))
ax.set_xlabel("pool sulfur content p, percent")
ax.set_ylabel("optimal profit, USD")
ax.set_title("Week 4, Figure 2: two separated hills, one nonconvex problem", fontsize=10)
ax.legend(fontsize=8, loc="upper right")
plt.show()

In [ ]:
# --- A global solver settles the question ------------------------------------
gopt = None
try:
    glob = pyo.SolverFactory("couenne")
    if glob is not None and glob.available(exception_flag=False):
        mg = build_haverly(p0=2.0)
        rg = glob.solve(mg)
        gopt = pyo.value(mg.profit)
        print(f"couenne (spatial branch and bound): termination "
              f"{rg.solver.termination_condition}, profit {gopt:,.4f}")
        print(f"   p = {pyo.value(mg.p):.4f}, A = {pyo.value(mg.A):.3f}, "
              f"B = {pyo.value(mg.B):.3f}, Py = {pyo.value(mg.Py):.3f}, "
              f"Cy = {pyo.value(mg.Cy):.3f}")
    else:
        print("no global solver available, falling back to the grid sweep")
except Exception as exc:
    print("global solver call failed:", exc)

reference = gopt if gopt is not None else float(np.nanmax(pprof))
print(f"\nreference global optimum : {reference:,.4f} USD")
print(f"best multistart result   : {ms.profit.max():,.4f} USD")
print(f"worst multistart result  : {ms.profit.min():,.4f} USD")
assert abs(reference - ms.profit.max()) < 1e-3, \
    "multistart did not reach the global optimum"

## 4. The p-formulation and the q-formulation

The p-formulation carries the pool quality `p` as a variable. The **q-formulation** (Ben-Tal and coauthors;
Tawarmalani and Sahinidis, Ch. 9) carries instead the *proportions* in which the pool is fed:

    q_i  =  fraction of the pool inlet contributed by feed i,      sum_i q_i = 1,   0 <= q_i <= 1

The feed flows are then `F_i = q_i sum_g P_g`, and the amount of feed `i` that reaches product `g` is
`q_i P_g`. Every bilinear term is now a product `q_i P_g` of a bounded proportion and a flow. For Haverly,

    max   9(Px + Cx) + 15(Py + Cy) - 6 qA (Px + Py) - 16 qB (Px + Py) - 10(Cx + Cy)

    s.t.  qA + qB = 1
          (3 qA + 1 qB) Px + 2 Cx <= 2.5 (Px + Cx)
          (3 qA + 1 qB) Py + 2 Cy <= 1.5 (Py + Cy)
          Px + Cx <= 100,   Py + Cy <= 200
          0 <= qA, qB <= 1,   all flows >= 0

The two formulations describe the same feasible set and the same global optimum, since `p = sum_i q_i s_i`.
They are not equivalent as *relaxations*, which is the whole reason for preferring one over the other.

### Why the difference shows up in a relaxation

Both models are nonconvex, so a solver that is expected to certify global optimality needs a convex
relaxation to bound them. The standard device is the **McCormick envelope**: replace each product `w = u v`
of variables with `u in [uL, uU]` and `v in [vL, vU]` by a new variable `w` together with

    w >= uL v + vL u - uL vL      w >= uU v + vU u - uU vU
    w <= uU v + vL u - uU vL      w <= uL v + vU u - uL vU

which is the convex hull of the graph of `u v` over the box. Relaxing every bilinear term this way turns the
pooling NLP into an LP whose optimum is a valid bound on the global optimum.

In the q-formulation the bilinear variables `v_ig = q_i P_g` also inherit the linear identity
`sum_i q_i = 1`. Multiplying that identity by `P_g` gives the valid equalities

    sum_i v_ig = P_g       for every product g

and multiplying it by the pool throughput limit gives `sum_g v_ig <= Cap q_i`. These are
**reformulation-linearization** (RLT) constraints: they are redundant for the nonlinear model but not for its
McCormick relaxation, and they are what makes the q-formulation (or pq-formulation) at least as tight as the
p-formulation. The comparison below measures the difference on two instances.

In [ ]:
# --- Haverly in the q-formulation --------------------------------------------
def build_haverly_q(qA0=0.5):
    m = pyo.ConcreteModel(name="haverly_q_formulation")
    m.qA = pyo.Var(bounds=(0.0, 1.0), initialize=qA0)
    m.qB = pyo.Var(bounds=(0.0, 1.0), initialize=1.0 - qA0)
    m.Px = pyo.Var(bounds=(0.0, 100.0), initialize=0.0)
    m.Py = pyo.Var(bounds=(0.0, 200.0), initialize=0.0)
    m.Cx = pyo.Var(bounds=(0.0, 100.0), initialize=0.0)
    m.Cy = pyo.Var(bounds=(0.0, 200.0), initialize=0.0)

    m.fractions = pyo.Constraint(expr=m.qA + m.qB == 1)
    m.spec_X = pyo.Constraint(expr=(3*m.qA + 1*m.qB)*m.Px + 2*m.Cx <= 2.5*(m.Px + m.Cx))
    m.spec_Y = pyo.Constraint(expr=(3*m.qA + 1*m.qB)*m.Py + 2*m.Cy <= 1.5*(m.Py + m.Cy))
    m.demand_X = pyo.Constraint(expr=m.Px + m.Cx <= 100)
    m.demand_Y = pyo.Constraint(expr=m.Py + m.Cy <= 200)

    m.profit = pyo.Objective(
        expr=9*(m.Px + m.Cx) + 15*(m.Py + m.Cy)
             - 6*m.qA*(m.Px + m.Py) - 16*m.qB*(m.Px + m.Py) - 10*(m.Cx + m.Cy),
        sense=pyo.maximize)
    return m

# multistart over the proportion of feed A, the q-formulation analogue of sweeping p
msq = []
for qA0 in np.linspace(0.0, 1.0, 9):
    mq = build_haverly_q(float(qA0))
    rq = nlp.solve(mq)
    assert rq.solver.termination_condition == pyo.TerminationCondition.optimal, \
        f"ipopt failed from qA0 = {qA0}"
    msq.append({"qA_start": round(float(qA0), 3),
                "profit": round(pyo.value(mq.profit), 4),
                "qA_final": round(pyo.value(mq.qA), 4),
                "implied p": round(3*pyo.value(mq.qA) + 1*pyo.value(mq.qB), 4)})
msq = pd.DataFrame(msq)
print(msq.to_string(index=False))
print("\ndistinct local optima:", sorted(msq.profit.unique(), reverse=True))
print(f"fraction of starting points reaching the global optimum: "
      f"{100*(msq.profit == msq.profit.max()).mean():.0f} percent   "
      f"(p-formulation: {100*(ms.profit == ms.profit.max()).mean():.0f} percent)")
assert abs(msq.profit.max() - ms.profit.max()) < 1e-3, \
    "both formulations must have the same global optimum"
print("\nSame global optimum, same two local solutions, same trap. Reparametrizing a nonconvex "
      "problem does not make it convex.")

In [ ]:
# --- One generic pooling builder, two formulations, exact or relaxed ---------
def pool_model(data, form="p", relax=False, init=None):
    """Single-pool problem. form='p' or 'q'; relax=True builds the McCormick/RLT LP."""
    F, G = data["feeds"], data["products"]
    s   = {i: F[i]["s"] for i in F}
    cst = {i: F[i]["c"] for i in F}
    sd, cd, cap = data["bypass"]["s"], data["bypass"]["c"], data["pool_cap"]
    U = {g: min(G[g]["dem"], cap) for g in G}          # valid upper bound on each pool outlet
    sL, sU = min(s.values()), max(s.values())

    m = pyo.ConcreteModel(name=f"pool_{form}{'_relaxed' if relax else ''}")
    m.P = pyo.Var(list(G), domain=pyo.NonNegativeReals, bounds=lambda m, g: (0.0, U[g]))
    m.D = pyo.Var(list(G), domain=pyo.NonNegativeReals, bounds=lambda m, g: (0.0, G[g]["dem"]))
    m.con = pyo.ConstraintList()

    if form == "p":
        m.f = pyo.Var(list(F), domain=pyo.NonNegativeReals, bounds=(0.0, cap))
        m.p = pyo.Var(bounds=(sL, sU), initialize=(init if init is not None else 0.5*(sL + sU)))
        m.con.add(sum(m.f[i] for i in F) == sum(m.P[g] for g in G))
        if relax:
            m.w = pyo.Var(list(G), domain=pyo.NonNegativeReals)      # w_g stands for p * P_g
            for g in G:
                m.con.add(m.w[g] >= sL*m.P[g])
                m.con.add(m.w[g] >= sU*m.P[g] + U[g]*m.p - sU*U[g])
                m.con.add(m.w[g] <= sU*m.P[g])
                m.con.add(m.w[g] <= sL*m.P[g] + U[g]*m.p - sL*U[g])
            out = {g: m.w[g] for g in G}
        else:
            out = {g: m.p*m.P[g] for g in G}
        m.con.add(sum(s[i]*m.f[i] for i in F) == sum(out[g] for g in G))
        pool_quality = {g: out[g] for g in G}
        feed_cost = sum(cst[i]*m.f[i] for i in F)
    else:
        m.q = pyo.Var(list(F), bounds=(0.0, 1.0),
                      initialize=(init if init is not None else 1.0/len(F)))
        m.con.add(sum(m.q[i] for i in F) == 1)
        if relax:
            m.v = pyo.Var(list(F), list(G), domain=pyo.NonNegativeReals)   # v_ig stands for q_i * P_g
            for i in F:
                for g in G:
                    m.con.add(m.v[i, g] >= m.P[g] + U[g]*m.q[i] - U[g])
                    m.con.add(m.v[i, g] <= m.P[g])
                    m.con.add(m.v[i, g] <= U[g]*m.q[i])
            for g in G:                                   # RLT: (sum_i q_i = 1) times P_g
                m.con.add(sum(m.v[i, g] for i in F) == m.P[g])
            for i in F:                                   # RLT: (sum_i q_i = 1) times the pool limit
                m.con.add(sum(m.v[i, g] for g in G) <= cap*m.q[i])
            vv = {(i, g): m.v[i, g] for i in F for g in G}
        else:
            vv = {(i, g): m.q[i]*m.P[g] for i in F for g in G}
        pool_quality = {g: sum(s[i]*vv[i, g] for i in F) for g in G}
        feed_cost = sum(cst[i]*vv[i, g] for i in F for g in G)

    for g in G:
        m.con.add(pool_quality[g] + sd*m.D[g] <= G[g]["spec"]*(m.P[g] + m.D[g]))
        m.con.add(m.P[g] + m.D[g] <= G[g]["dem"])
    m.con.add(sum(m.P[g] for g in G) <= cap)
    m.profit = pyo.Objective(
        expr=sum(G[g]["price"]*(m.P[g] + m.D[g]) for g in G)
             - feed_cost - cd*sum(m.D[g] for g in G), sense=pyo.maximize)
    return m

def pool_lp_fixed_quality(data, p):
    """With the pool quality fixed the single-pool problem is an ordinary LP."""
    F, G = data["feeds"], data["products"]
    s   = {i: F[i]["s"] for i in F}
    cst = {i: F[i]["c"] for i in F}
    sd, cd, cap = data["bypass"]["s"], data["bypass"]["c"], data["pool_cap"]
    m = pyo.ConcreteModel()
    m.P = pyo.Var(list(G), domain=pyo.NonNegativeReals)
    m.D = pyo.Var(list(G), domain=pyo.NonNegativeReals)
    m.f = pyo.Var(list(F), domain=pyo.NonNegativeReals)
    m.con = pyo.ConstraintList()
    m.con.add(sum(m.f[i] for i in F) == sum(m.P[g] for g in G))
    m.con.add(sum(s[i]*m.f[i] for i in F) == p*sum(m.P[g] for g in G))
    for g in G:
        m.con.add(p*m.P[g] + sd*m.D[g] <= G[g]["spec"]*(m.P[g] + m.D[g]))
        m.con.add(m.P[g] + m.D[g] <= G[g]["dem"])
    m.con.add(sum(m.P[g] for g in G) <= cap)
    m.profit = pyo.Objective(expr=sum(G[g]["price"]*(m.P[g] + m.D[g]) for g in G)
                             - sum(cst[i]*m.f[i] for i in F)
                             - cd*sum(m.D[g] for g in G), sense=pyo.maximize)
    r = lp.solve(m)
    return (pyo.value(m.profit)
            if r.solver.termination_condition == pyo.TerminationCondition.optimal else np.nan)

HAVERLY = {"feeds": {"A": {"s": 3.0, "c": 6.0}, "B": {"s": 1.0, "c": 16.0}},
           "bypass": {"s": 2.0, "c": 10.0}, "pool_cap": 300.0,
           "products": {"X": {"price": 9.0, "spec": 2.5, "dem": 100.0},
                        "Y": {"price": 15.0, "spec": 1.5, "dem": 200.0}}}

# a second instance: four feeds into a pool whose tank throughput is limited
EXTENDED = {"feeds": {"A": {"s": 3.0, "c": 6.0},  "B": {"s": 1.0, "c": 16.0},
                      "C": {"s": 2.5, "c": 8.0},  "D": {"s": 0.5, "c": 20.0}},
            "bypass": {"s": 2.0, "c": 10.0}, "pool_cap": 150.0,
            "products": {"X": {"price": 9.0, "spec": 2.5, "dem": 100.0},
                         "Y": {"price": 15.0, "spec": 1.5, "dem": 200.0}}}

# sanity check: the generic builder reproduces the hand-written Haverly model
m_check = pool_model(HAVERLY, form="p", relax=False, init=1.0)
r_check = nlp.solve(m_check)
assert r_check.solver.termination_condition == pyo.TerminationCondition.optimal
print(f"generic p-model on Haverly from p0 = 1.0 : {pyo.value(m_check.profit):.4f} USD "
      f"(hand-written model gave {low['profit']:.4f})")
assert abs(pyo.value(m_check.profit) - low["profit"]) < 1e-2

In [ ]:
# --- Global optimum of each instance, two independent ways -------------------
INSTANCES = [("Haverly (2 feeds, no pool limit)", HAVERLY),
             ("extended (4 feeds, pool <= 150)", EXTENDED)]

glob = pyo.SolverFactory("couenne")
HAS_GLOBAL = glob is not None and glob.available(exception_flag=False)

GLOBAL_OPT = {}
for name, data in INSTANCES:
    sL = min(v["s"] for v in data["feeds"].values())
    sU = max(v["s"] for v in data["feeds"].values())
    grid = np.linspace(sL, sU, 201)
    sweep = np.array([pool_lp_fixed_quality(data, float(p)) for p in grid])
    best_sweep = float(np.nanmax(sweep))
    if HAS_GLOBAL:
        mg = pool_model(data, form="p", relax=False)
        rg = glob.solve(mg)
        assert rg.solver.termination_condition == pyo.TerminationCondition.optimal, \
            f"couenne failed on {name}"
        best_global = pyo.value(mg.profit)
        assert abs(best_global - best_sweep) < 1e-3, \
            f"the sweep and the global solver disagree on {name}"
    else:
        best_global = best_sweep
    GLOBAL_OPT[name] = best_global
    print(f"{name:35s} sweep {best_sweep:8.4f}   couenne {best_global:8.4f} "
          f"at p = {grid[int(np.nanargmax(sweep))]:.4f}")

# the q-formulation must reach the same value; check it by multistart on the harder instance
best_q = max(
    (pyo.value(mq_.profit) for mq_ in
     (pool_model(EXTENDED, form="q", relax=False, init=float(a)) for a in np.linspace(0.05, 0.95, 7))
     if nlp.solve(mq_).solver.termination_condition == pyo.TerminationCondition.optimal))
print(f"\nbest q-formulation multistart value on the extended instance: {best_q:.4f}")
assert abs(best_q - GLOBAL_OPT["extended (4 feeds, pool <= 150)"]) < 1e-2, \
    "the q-formulation should reach the same global optimum"

In [ ]:
# --- McCormick relaxation bound of each formulation --------------------------
rows = []
for name, data in INSTANCES:
    for form in ("p", "q"):
        mrx = pool_model(data, form=form, relax=True)
        rrx = lp.solve(mrx)
        assert rrx.solver.termination_condition == pyo.TerminationCondition.optimal, \
            f"relaxation of the {form}-formulation of {name} did not solve"
        bound = pyo.value(mrx.profit)
        rows.append({"instance": name, "formulation": form,
                     "global optimum": round(GLOBAL_OPT[name], 4),
                     "relaxation bound": round(bound, 4),
                     "bound gap percent": round(100*(bound - GLOBAL_OPT[name])/GLOBAL_OPT[name], 3),
                     "relaxation rows": mrx.nconstraints(),
                     "relaxation cols": mrx.nvariables()})
pq = pd.DataFrame(rows)
print(pq.to_string(index=False))

for name in GLOBAL_OPT:
    sub = pq[pq["instance"] == name]
    for _, r_ in sub.iterrows():
        assert r_["relaxation bound"] >= GLOBAL_OPT[name] - 1e-6, "a relaxation bound must be valid"
assert pq.loc[3, "relaxation bound"] < pq.loc[2, "relaxation bound"] - 1e-4, \
    "on the extended instance the q-formulation should give the strictly tighter bound"
print("\nEvery bound is valid, and on the extended instance the q-formulation bound is strictly tighter.")

In [ ]:
# --- Figure 3: what each relaxation certifies --------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.7), constrained_layout=True)
inst = list(dict.fromkeys(pq["instance"]))

for ax, name in zip(axes, inst):
    tidy(ax)
    sub = pq[pq["instance"] == name].reset_index(drop=True)
    xpos = np.arange(len(sub)); w = 0.38
    ax.bar(xpos - w/2, sub["relaxation bound"], width=w, color=PALETTE[1],
           label="McCormick relaxation bound")
    ax.bar(xpos + w/2, sub["global optimum"], width=w, color=PALETTE[0],
           label="global optimum (couenne)")
    for k in range(len(sub)):
        ax.text(xpos[k] - w/2, sub["relaxation bound"][k] + 8, f"{sub['relaxation bound'][k]:,.1f}",
                ha="center", fontsize=7.5, color=INK)
        ax.text(xpos[k] + w/2, sub["global optimum"][k] + 8, f"{sub['global optimum'][k]:,.1f}",
                ha="center", fontsize=7.5, color=INK)
        ax.annotate(f"{sub['bound gap percent'][k]:.1f} % above", (xpos[k], sub["relaxation bound"][k] + 42),
                    ha="center", fontsize=7.5, color=PALETTE[2])
    ax.set_xticks(xpos)
    ax.set_xticklabels([f"{f}-formulation\n({r} rows)" for f, r in
                        zip(sub["formulation"], sub["relaxation rows"])], fontsize=8)
    ax.set_ylabel("profit, USD")
    ax.set_ylim(0, sub["relaxation bound"].max()*1.45)
    ax.set_title(name, fontsize=9.5)
axes[0].legend(fontsize=8, loc="upper right")
fig.suptitle("Week 4, Figure 3: the q-formulation buys a tighter certificate with extra rows",
             color=INK, fontsize=10)
plt.show()

## 5. Interpretation

**The linear blending LP.** The optimum is 413,439.75 USD on 4,080.60 t of Regular and 5,000.00 t of
Premium. All six quality specifications bind at the optimum: Regular sits exactly at RON 91, RVP 9.0 kPa and
290 ppm sulfur, Premium exactly at RON 95, RVP 8.5 kPa and 190 ppm. That is normal and it is the point of a
blending model. Quality that exceeds specification is quality that has been given away, so the optimizer
gives away nothing. Reformate and FCC naphtha are exhausted and carry positive availability duals, which say
what the refinery would pay for one more tonne. Alkylate, butane and straight run have slack and a dual of
zero.

**The linearization is exact, and it is worth checking.** Solving the same model with the specifications left
as explicit ratios and handing it to ipopt returns the same optimum to within solver tolerance, while the
constraints report a polynomial degree of `None` instead of 1. The linear form is not an approximation of the
ratio form, it is the same feasible set intersected with `y_g > 0`. The caveat is the shut-off case: at
`y_g = 0` the linear constraint reads `0 >= 0` and is satisfied by any recipe, while the ratio is undefined.
A model in which a grade may be switched off therefore needs a binary variable, not a ratio.

**A ratio objective is a different objective.** Maximizing margin per tonne through the Charnes-Cooper LP
gives 45.8708 USD/t on a plan of 4,000.00 t of Regular and 5,000.00 t of Premium, whose total margin is
412,837.10 USD. That is 602.65 USD less than the total-margin optimum. The plant that maximizes the ratio
deliberately declines the last 80.6 t of Regular because that increment earns less than the current average.
Which objective is correct is a business question, not a mathematical one, but the two must not be confused.

**Where the pool changes everything.** The pooling instance is small, has six flow variables and one quality
variable, and is completely nonconvex. From a starting pool quality of 1.0 ipopt converges to a profit of
400.00 with `B = 100`, `Py = 100`, `Cy = 100`. From a starting quality of 3.0 the same solver, on the same
model, converges to 100.00 with `A = 50`, `Px = 50`, `Cx = 50`. Both runs report termination status
`optimal`, because both points genuinely satisfy the first-order optimality conditions. The status word from
a local NLP solver is a statement about the Karush-Kuhn-Tucker conditions, not about global optimality.

**The multistart table shows the split cleanly.** Starting values of `p` at or below 1.75 reach 400, values
at or above 2.0 reach 100. Accepting the wrong one costs 300 USD out of 400, or 75 percent of the achievable
margin, on a problem with seven variables.

**Figure 2 explains why.** Fixing `p` turns the pooling problem back into an LP, so sweeping `p` over its
range traces the true profit profile. That profile has two separated hills with a flat zero valley over
roughly `1.53 <= p <= 2.40` on the sweep grid, where no feasible plan makes any money. A gradient-based
method climbs the hill it starts on and cannot cross the valley. Nothing is wrong with ipopt: the problem,
not the solver, is nonconvex.

**p against q.** The q-formulation is a change of variables, not a change of problem: it has the same global
optimum of 400 and the same pair of local solutions, and a multistart over the initial proportion `qA` splits
into the same two basins. What changes is the relaxation. On the Haverly instance the McCormick bounds of the
two formulations coincide at 500, 25 percent above the global optimum, because a single pool fed by two
streams and limited only by product demand leaves the RLT constraints nothing to add. On the extended
instance, four feeds and a pool throughput limit of 150 units, the p-formulation bound is 555.56 and the
q-formulation bound is 466.67 against a global optimum of 400, so the gap falls from 38.9 percent to 16.7
percent for the cost of extra rows. That is the practical reason global solvers build the pq-formulation
internally: the bound is what a spatial branch-and-bound tree spends its time closing.

**Practical consequences.**

- Never report a single local NLP solve as *the* optimum of a nonconvex problem. Report the starting point,
  and run a multistart.
- Prefer a physical starting point. Here, initializing the pool quality from the actual current plant
  operation would decide the answer.
- Use a global solver (couenne, BARON, ANTIGONE) when the problem is small, and report the relaxation bound
  alongside the incumbent so that the reader knows how much room is left.
- Choose the formulation with the tighter relaxation before blaming the solver.

Discrete operating reality (fixed charges for lining up a tank, minimum run rates, cardinality limits on how
many streams may run at once, and the implications between them) is deferred to Week 5, where it is treated
with binary variables, big-M and convex hull formulations.

## 6. Exercises

**Exercise 1 (introductory).** A new low-sulfur component, isomerate, becomes available: RON 87, RVP 14 kPa,
sulfur 2 ppm, cost 735 USD/t, up to 1,500 t. Add it to the linear blending LP, re-solve, and report the new
margin, how much isomerate is used, and which previously binding specification stops binding. Confirm your
prediction of its usefulness beforehand using the availability and specification duals of the base solution.

**Exercise 2 (introductory).** The Regular sulfur specification tightens from 290 to 150 ppm. Re-solve the
LP, report the loss of margin, and compare it with the prediction obtained from the sulfur dual in the base
solution multiplied by the change of 140 ppm. Explain why the two numbers differ.

**Exercise 3 (intermediate).** A downstream unit requires that the recycle ratio `R / (R + F)` never exceed
0.35 and that the sulfur of the combined stream `(s_R R + s_F F) / (R + F)` stay below 120 ppm, with
`s_R = 260` and `s_F = 40` ppm. Write both requirements as linear inequalities in `R` and `F`, state the
condition under which the transformation is valid, and plot the resulting feasible region in the `(F, R)`
plane for `0 <= F <= 100`. Then explain what changes if the unit may be shut down entirely.

**Exercise 4 (intermediate).** Run a proper multistart on the Haverly problem. Draw 100 random starting
points with `p` uniform on `[1, 3]` and each flow uniform on `[0, 100]`, solve with ipopt from each, and
report the histogram of terminal objective values, the fraction reaching 400, and the worst value found.
Then repeat with the pool quality fixed at the current plant value of 2.2 and comment on what a plant
engineer should conclude.

**Exercise 5 (advanced).** Convexify the Haverly p-formulation by hand. Replace each bilinear term `p * P` by
a new variable `w` and add the four McCormick envelope inequalities built from the bounds `1 <= p <= 3` and
`0 <= P <= 200`, without using `pool_model`. Solve the resulting LP and check that your bound matches the
500.00 reported in section 4. Then tighten the bound by bisecting the interval for `p` at `p = 2` and solving
the two resulting relaxations. Report the better of the two bounds, explain why the maximum over the two
subintervals is still valid, and relate the procedure to spatial branch and bound.

**Exercise 6 (introductory, cross-tool).** Build the linear blending model in
`excel/W04_blending_STUDENT.xlsx` with the layout given above, solve it with
OpenSolver, and confirm that objective cell `C45` equals the maximum blending margin printed in Section
1 to the cent and that the changing block `C20:D24` equals the recipe table component by component. Then
build the Haverly instance on a second sheet, with the pool sulfur balance written as a product of two
changing cells, and run Solver twice, starting once from a pool quality of 1.0 and once from 3.0.
Reproduce both of the objective values that Section 3 obtains from its two starting points, record the
message Solver displays in each case, and state in two sentences what a user would have concluded if
only the first run had been made.


## 7. Takeaways

- A quality specification is naturally a ratio constraint. Multiplying by the blend volume, which is itself a variable of the same balance, makes it linear. That single manipulation is why blending is an LP.
- The multiply-through rule is exact only when the denominator is strictly positive and lives in the same balance as the numerator. If the denominator can be zero, the linear form is a relaxation and the honest model is a disjunction. If the denominator belongs to another balance, the nonlinearity is real.
- A ratio objective is a linear-fractional program. The Charnes-Cooper substitution turns it into an LP, and the resulting plan is generally not the plan that maximizes the total.
- A pool forces every product drawn from it to see the same composition. The pool quality then multiplies flows that belong to other balances, bilinear terms appear, and the feasible set stops being convex.
- A local NLP solver reporting `optimal` is asserting the Karush-Kuhn-Tucker conditions, not global optimality. On the Haverly instance ipopt returns 400 or 100 depending only on the starting point, a 75 percent difference on seven variables.
- The p- and q-formulations describe the same problem and the same local solutions, but not the same relaxation. Adding the RLT constraints of the q-formulation cut the bound gap on the extended instance from 38.9 percent to 16.7 percent, which is the quantity a global solver actually works to close.
- Report the relaxation bound with the incumbent. An answer to a nonconvex problem without a bound is an opinion.